In [0]:
import os, sys

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/".join(nb_path.split("/")[:-2])          # .../notebooks
if not repo_root.startswith("/Workspace"):
    repo_root = "/Workspace" + repo_root
sys.path.append(os.path.join(repo_root, "src"))

from retail_sales.contracts import BRONZE_SALES_SCHEMA, SILVER_SALES_SCHEMA, schema_problems

dbutils.widgets.text("catalog", "retail_sales_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
CHECKS = {
    f"{catalog}.bronze.sales_raw":     BRONZE_SALES_SCHEMA,
    f"{catalog}.silver.sales_cleaned": SILVER_SALES_SCHEMA,
}

failures = {}
for table, contract in CHECKS.items():
    problems = schema_problems(spark.table(table), contract)
    print(f"{table}: {'OK' if not problems else 'FAILED'}")
    for p in problems:
        print(f"   - {p}")
    if problems:
        failures[table] = problems

assert not failures, f"Schema contract violated for: {', '.join(failures)}"